# Preparación de Datos: Proyecto MASTER (Target Suavizado)

Este notebook se encarga de la recolección de datos y la ingeniería de características para el modelo de predicción de acciones de JPMorgan Chase (JPM). Para obtener un modelo más predictible ($R^2$ alto), predeciremos el precio crudo suavizado (Promedio Móvil de 5 días) en lugar del retorno diario ruidoso.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraciones visuales iniciales
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

### 1. Recolección de Datos
Descarga de la información diaria usando la API de Yahoo Finance.

In [2]:
# Parámetros de descarga requeridos
tickers = ['JPM', 'BAC', 'C', 'WFC', 'XLF']
start_date = '2010-01-01'
end_date = '2026-07-05'

print("Descargando datos históricos...")
raw_data = yf.download(tickers, start=start_date, end=end_date)

# Extracción de características clave
adj_close = raw_data['Close']
volume = raw_data['Volume']

print(f"Datos descargados. Dimensiones: {adj_close.shape}")

Descargando datos históricos...


[*********************100%***********************]  5 of 5 completed

Datos descargados. Dimensiones: (4149, 5)


### 2. Estacionariedad: Retornos Logarítmicos
Cálculo de la tasa de crecimiento continuo para garantizar la estacionariedad de las series de tiempo (usados como contexto, aunque nuestro target final será el precio).

In [3]:
# Cálculo de retornos logarítmicos: r_t = ln(P_t / P_{t-1})
log_returns = np.log(adj_close / adj_close.shift(1))

# Variación del volumen de transacciones
volume_pct_change = volume.pct_change()

print("Cálculos de estacionariedad completados.")

Cálculos de estacionariedad completados.


### 3. Feature Engineering: Momentum, Volatilidad y Objetivo (Target)
Inyección de heurísticas financieras y cálculo del Promedio Móvil Objetivo.

In [4]:
# Estructura para el consolidado final de características
features = pd.DataFrame(index=adj_close.index)
window_volatility = 10

for ticker in tickers:
    # Series base estacionarias
    features[f'{ticker}_Log_Ret'] = log_returns[ticker]
    features[f'{ticker}_Vol_Pct'] = volume_pct_change[ticker]
    
    # Volatilidad rodante (10 días)
    features[f'{ticker}_Volatility'] = log_returns[ticker].rolling(window=window_volatility).std()
    
    # RSI (Relative Strength Index) - 14 días
    features[f'{ticker}_RSI'] = ta.momentum.RSIIndicator(close=adj_close[ticker], window=14).rsi()
    
    # MACD (Moving Average Convergence Divergence)
    macd = ta.trend.MACD(close=adj_close[ticker])
    features[f'{ticker}_MACD'] = macd.macd()
    features[f'{ticker}_MACD_Diff'] = macd.macd_diff()

# NUEVO TARGET: Precio crudo y Promedio Móvil Suavizado de 5 días de JPM
features['JPM_Close'] = adj_close['JPM']
# Calculamos el promedio de los PRÓXIMOS 5 días (shift(-5) empuja el futuro hacia hoy para que el modelo lo aprenda)
features['JPM_Target_MA5'] = adj_close['JPM'].rolling(window=5).mean().shift(-5)

print("Características técnicas generadas. Target JPM_Target_MA5 creado.")

Características técnicas generadas. Target JPM_Target_MA5 creado.


### 4. Codificación Temporal (Time Embeddings)
Preservación de la estacionalidad usando transformaciones trigonométricas.

In [5]:
# Extracción de componentes temporales del índice
day_of_week = features.index.dayofweek
month = features.index.month

# Proyección seno/coseno para el día de la semana (ciclo de 5 días hábiles)
features['Day_Sin'] = np.sin(2 * np.pi * day_of_week / 5)
features['Day_Cos'] = np.cos(2 * np.pi * day_of_week / 5)

# Proyección seno/coseno para el mes (ciclo de 12 meses)
features['Month_Sin'] = np.sin(2 * np.pi * month / 12)
features['Month_Cos'] = np.cos(2 * np.pi * month / 12)

print("Codificaciones temporales aplicadas.")

Codificaciones temporales aplicadas.


### 5. Consolidación, Verificación y Exportación
Ensamblaje del dataset final, manejo de valores nulos (ahora hay nulos al final por el `shift(-5)`) y guardado en disco.

In [6]:
# Eliminación de filas con valores nulos generados por las ventanas de cálculo pasadas y futuras
final_dataset = features.dropna()

print(f"Dimensiones finales del dataset: {final_dataset.shape}")

# Guardar en disco
final_dataset.to_csv('processed_market_data.csv')
print("¡Datos guardados exitosamente en 'processed_market_data.csv'!")

Dimensiones finales del dataset: (4111, 36)
¡Datos guardados exitosamente en 'processed_market_data.csv'!
